# Task 2: Custom Byte-Pair Encoding (BPE) Tokenizer & Autoregressive Causal LM

## Objective

To understand the mechanics of subword tokenization and autoregressive sequence generation by building a custom Byte-Pair Encoding (BPE) tokenizer and a causal language model training loop from the ground up.

## Technologies / Tools Used

- Python
- PyTorch
- NumPy
- Google Colab

## Step 1: Import Required Libraries

Import the libraries required for BPE tokenization and model training.

In [24]:
import numpy as np
import torch
import torch.nn as nn

np.random.seed(42)
torch.manual_seed(42)

## Step 2: Create Training Corpus

Create a small custom text corpus for learning BPE tokens and training the language model.

In [25]:
corpus = """
generative artificial intelligence is transforming technology
artificial intelligence can generate text images and code
language models learn patterns from large amounts of text
generative models can create new content
"""

corpus = corpus.lower().strip()

print(corpus)

generative artificial intelligence is transforming technology
artificial intelligence can generate text images and code
language models learn patterns from large amounts of text
generative models can create new content


## Step 3: Prepare Text for BPE

Represent each word as a sequence of characters and add an end-of-word token.

In [26]:
words = corpus.split()

word_tokens = {}

for word in words:
    word_tokens[word] = list(word) + ["</w>"]

print(list(word_tokens.items())[:3])

[('generative', ['g', 'e', 'n', 'e', 'r', 'a', 't', 'i', 'v', 'e', '</w>']), ('artificial', ['a', 'r', 't', 'i', 'f', 'i', 'c', 'i', 'a', 'l', '</w>']), ('intelligence', ['i', 'n', 't', 'e', 'l', 'l', 'i', 'g', 'e', 'n', 'c', 'e', '</w>'])]


## Step 4: Count Token Pairs

Count the frequency of adjacent token pairs in the corpus.

In [27]:
def get_pair_counts(word_tokens):

    pair_counts = {}

    for tokens in word_tokens.values():

        for i in range(len(tokens) - 1):

            pair = (
                tokens[i],
                tokens[i + 1]
            )

            pair_counts[pair] = (
                pair_counts.get(pair, 0) + 1
            )

    return pair_counts


pair_counts = get_pair_counts(word_tokens)

print(
    sorted(
        pair_counts.items(),
        key=lambda x: x[1],
        reverse=True
    )[:10]
)

[(('e', '</w>'), 7), (('t', 'e'), 7), (('g', 'e'), 6), (('s', '</w>'), 5), (('e', 'n'), 4), (('a', 't'), 4), (('n', 't'), 4), (('a', 'n'), 4), (('n', 'e'), 3), (('e', 'r'), 3)]


## Step 5: Merge the Most Frequent Pair

Merge the most frequently occurring token pair into a single token.

In [28]:
def merge_pair(word_tokens, pair):

    new_token = pair[0] + pair[1]
    result = {}

    for word, tokens in word_tokens.items():

        new_tokens = []
        i = 0

        while i < len(tokens):

            if (
                i < len(tokens) - 1
                and tokens[i] == pair[0]
                and tokens[i + 1] == pair[1]
            ):

                new_tokens.append(new_token)
                i += 2

            else:

                new_tokens.append(tokens[i])
                i += 1

        result[word] = new_tokens

    return result

## Step 6: Build BPE Vocabulary

Repeatedly merge the most frequent token pairs to construct the BPE vocabulary.

In [29]:
num_merges = 20
merges = []

for _ in range(num_merges):

    pair_counts = get_pair_counts(word_tokens)

    if not pair_counts:
        break

    best_pair = max(
        pair_counts,
        key=pair_counts.get
    )

    if pair_counts[best_pair] < 2:
        break

    word_tokens = merge_pair(
        word_tokens,
        best_pair
    )

    merges.append(best_pair)

print("Number of merges:", len(merges))
print("Learned merges:", merges)

Number of merges: 16
Learned merges: [('e', '</w>'), ('t', 'e'), ('s', '</w>'), ('g', 'e'), ('a', 't'), ('a', 'n'), ('ge', 'n'), ('a', 'r'), ('gen', 'e'), ('gene', 'r'), ('gener', 'at'), ('i', 'n'), ('t', '</w>'), ('c', 'o'), ('g', 'e</w>'), ('m', 'o')]


## Step 7: Create BPE Tokenizer

Apply the learned BPE merge rules to tokenize new text.

In [30]:
def tokenize_word(word, merges):

    tokens = list(word) + ["</w>"]

    for pair in merges:

        new_tokens = []
        i = 0

        while i < len(tokens):

            if (
                i < len(tokens) - 1
                and tokens[i] == pair[0]
                and tokens[i + 1] == pair[1]
            ):

                new_tokens.append(
                    pair[0] + pair[1]
                )

                i += 2

            else:

                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens

    return tokens


def tokenize(text):

    tokens = []

    for word in text.lower().split():
        tokens.extend(
            tokenize_word(word, merges)
        )

    return tokens


print(tokenize("artificial intelligence"))

['ar', 't', 'i', 'f', 'i', 'c', 'i', 'a', 'l', '</w>', 'in', 'te', 'l', 'l', 'i', 'gen', 'c', 'e</w>']


## Step 8: Convert Tokens to IDs

Create numerical IDs for the BPE tokens so they can be processed by PyTorch.

In [31]:
final_tokens = set()

for tokens in word_tokens.values():
    final_tokens.update(tokens)

vocab = [
    "<PAD>",
    "<UNK>"
] + sorted(final_tokens)

token_to_id = {
    token: i
    for i, token in enumerate(vocab)
}

id_to_token = {
    i: token
    for token, i in token_to_id.items()
}

print("Vocabulary size:", len(vocab))

Vocabulary size: 38


## Step 9: Encode Training Data

Convert the training corpus into numerical token IDs.

In [32]:
def encode(text):

    tokens = tokenize(text)

    return [
        token_to_id.get(
            token,
            token_to_id["<UNK>"]
        )
        for token in tokens
    ]


encoded_data = encode(corpus)

data = torch.tensor(
    encoded_data,
    dtype=torch.long
)

print("Encoded data:")
print(data)

Encoded data:
tensor([17, 19, 34, 11,  5, 30, 19, 12, 19,  7, 19,  3, 21,  2, 20, 32, 21, 21,
        19, 16,  7, 11, 19, 29, 30, 27,  4, 28, 12, 25, 27, 22, 20, 13,  2, 32,
         7, 18, 24, 25, 21, 25, 13, 37,  2,  5, 30, 19, 12, 19,  7, 19,  3, 21,
         2, 20, 32, 21, 21, 19, 16,  7, 11,  7,  4,  2, 17, 11, 32, 36, 31, 19,
        22,  3, 14, 29,  4,  9,  2,  8,  9, 11, 21,  4, 13, 33,  3, 15, 23,  9,
        10, 21, 29, 21, 10,  5, 24,  2, 26,  6, 32, 27, 24, 29, 12, 27, 25, 22,
         2, 21,  5, 15,  3, 23, 33, 24, 30, 29, 25, 12,  2, 32, 36, 31, 17, 19,
        34, 11, 23,  9, 10, 21, 29,  7,  4,  2,  7, 27, 10,  6, 11, 24, 10, 35,
         2,  8, 24, 32, 24, 31])


## Step 10: Create Input and Target Sequences

Shift the target sequence by one position so that the model learns next-token prediction.

In [33]:
sequence_length = 8

X = data[:-1]
Y = data[1:]

size = (
    len(X) // sequence_length
) * sequence_length

X = X[:size].view(
    -1,
    sequence_length
)

Y = Y[:size].view(
    -1,
    sequence_length
)

print("Input shape:", X.shape)
print("Target shape:", Y.shape)

Input shape: torch.Size([18, 8])
Target shape: torch.Size([18, 8])


## Step 11: Create Causal Mask

Create a causal mask to prevent the model from accessing future tokens.

In [34]:
causal_mask = torch.triu(
    torch.ones(
        sequence_length,
        sequence_length
    ),
    diagonal=1
)

causal_mask = causal_mask.masked_fill(
    causal_mask == 1,
    float("-inf")
)

causal_mask = causal_mask.masked_fill(
    causal_mask == 0,
    0.0
)

print(causal_mask)

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])


## Step 12: Define the Causal Language Model

Create a small Transformer-based causal language model using PyTorch.

In [35]:
class CausalLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=64,
        num_heads=4
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.position = nn.Embedding(
            sequence_length,
            embedding_dim
        )

        layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=2
        )

        self.output = nn.Linear(
            embedding_dim,
            vocab_size
        )

    def forward(self, x):

        positions = torch.arange(
            x.size(1)
        )

        x = (
            self.embedding(x)
            + self.position(positions)
        )

        mask = torch.triu(
            torch.ones(
                x.size(1),
                x.size(1)
            ),
            diagonal=1
        ).masked_fill(
            torch.triu(
                torch.ones(
                    x.size(1),
                    x.size(1)
                ),
                diagonal=1
            ) == 1,
            float("-inf")
        )

        x = self.transformer(
            x,
            mask=mask
        )

        return self.output(x)

## Step 13: Initialize the Model

Initialize the model, optimizer, and loss function.

In [36]:
model = CausalLM(len(vocab))

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss()

print("Model initialized successfully.")

Model initialized successfully.


## Step 14: Train the Model

Train the causal language model to predict the next token.

In [37]:
epochs = 100

for epoch in range(epochs):

    optimizer.zero_grad()

    logits = model(X)

    loss = criterion(
        logits.reshape(-1, len(vocab)),
        Y.reshape(-1)
    )

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch {epoch + 1}, "
            f"Loss: {loss.item():.4f}"
        )

Epoch 10, Loss: 1.9855
Epoch 20, Loss: 1.0627
Epoch 30, Loss: 0.5549
Epoch 40, Loss: 0.2974
Epoch 50, Loss: 0.1973
Epoch 60, Loss: 0.1374
Epoch 70, Loss: 0.1167
Epoch 80, Loss: 0.1017
Epoch 90, Loss: 0.0945
Epoch 100, Loss: 0.0793


## Step 15: Generate Text

Generate new tokens autoregressively using the trained model.

In [38]:
def generate(
    model,
    input_ids,
    max_new_tokens=10
):

    model.eval()

    generated = input_ids.clone()

    with torch.no_grad():

        for _ in range(max_new_tokens):

            current = generated[
                :, -sequence_length:
            ]

            logits = model(current)

            next_token = torch.argmax(
                logits[:, -1, :],
                dim=-1,
                keepdim=True
            )

            generated = torch.cat(
                [generated, next_token],
                dim=1
            )

    return generated

## Step 16: Test Text Generation

Provide a prompt and generate new tokens.

In [39]:
prompt = "artificial intelligence"

prompt_ids = encode(prompt)

prompt_tensor = torch.tensor(
    [prompt_ids],
    dtype=torch.long
)

generated_ids = generate(
    model,
    prompt_tensor,
    max_new_tokens=10
)

generated_tokens = [
    id_to_token.get(
        i,
        "<UNK>"
    )
    for i in generated_ids[0].tolist()
]

print("Generated tokens:")
print(generated_tokens)


Generated tokens:
['ar', 't', 'i', 'f', 'i', 'c', 'i', 'a', 'l', '</w>', 'in', 'te', 'l', 'l', 'i', 'gen', 'c', 'e</w>', 'mo', 'd', 'e', 'l', '</w>', 'c', 'an', '</w>', 'c', 'an']


## Conclusion

The custom BPE tokenizer and autoregressive causal language model were successfully implemented using Python, PyTorch, and NumPy. The implementation includes BPE pair merging, vocabulary construction, token encoding, causal masking, model training, and autoregressive text generation.